In [2]:
from pygbif import species, occurrences as occ
import pandas as pd

/Users/victoriacolombo/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


## Drive data

Let's take a look at the data uploaded to Google Drive at !2026-NABU-Asian-Hornet-Project > 01-Data

In [3]:
a_data = pd.read_csv('/Users/victoriacolombo/Documents/nabu/asian_hornet_DE.csv')
e_data= pd.read_csv('/Users/victoriacolombo/Documents/nabu/european_hornet_DE.csv')

/var/folders/xn/08v6g4r14lg5tpt_n37xl_5c0000gn/T/ipykernel_3155/1054268940.py:1: DtypeWarning: Columns (69,71,72,73,74,78,103) have mixed types. Specify dtype option on import or set low_memory=False.
  a_data = pd.read_csv('/Users/victoriacolombo/Documents/nabu/asian_hornet_DE.csv')
/var/folders/xn/08v6g4r14lg5tpt_n37xl_5c0000gn/T/ipykernel_3155/1054268940.py:2: DtypeWarning: Columns (75,81,82,95,101,103,104,105,109,110,111,113,114,115,116,117,118,119,120,121,122,123,126,127,128,129,130,132,133,134,135,136,137,138,139,140) have mixed types. Specify dtype option on import or set low_memory=False.
  e_data= pd.read_csv('/Users/victoriacolombo/Documents/nabu/european_hornet_DE.csv')


### 1. Size and date range

- The datasets **differ** in the amount of columns
- I'm using the column **eventDate** for the date information. 
There's also **dateIdentified** (both datasets) and **verbatimEventDate** (only european hornet)
- **eventDate** has different formats in both datasets
- the date order format is yyyy-mm-dd

In [ ]:
print('Asian hornet data shape:', a_data.shape, '\nEuropean hornet data shape:', 
      e_data.shape, '\n\nAsian hornets date range:', a_data.eventDate.min(), 'to', a_data.eventDate.max(), 
      '\nEuropean hornets date range:', e_data.eventDate.min(), 'to', e_data.eventDate.max())


Asian hornet data shape: (17866, 117) 
European hornet data shape: (61236, 141) 

Asian hornets date range: 2014-09-08 to 2025-12-30 
European hornets date range: 2000-05-05T00:00 to 2025-12-29T00:00


In [9]:
print('first date:',a_data.eventDate[0], '\n first date\'s month:',a_data.month[0])

first date: 2014-09-08 
 first date's month: 9


In [20]:
print(a_data.duplicated().sum())

0


In [21]:
print(e_data.duplicated().sum())

0


### 2. Possible relevant columns, types, value counts, and missing values 

In [10]:
a_data.info(verbose=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17866 entries, 0 to 17865
Data columns (total 117 columns):
 #    Column                                 Dtype  
---   ------                                 -----  
 0    key                                    int64  
 1    datasetKey                             object 
 2    publishingOrgKey                       object 
 3    installationKey                        object 
 4    hostingOrganizationKey                 object 
 5    publishingCountry                      object 
 6    protocol                               object 
 7    lastCrawled                            object 
 8    lastParsed                             object 
 9    crawlId                                int64  
 10   extensions                             object 
 11   basisOfRecord                          object 
 12   occurrenceStatus                       object 
 13   classifications                        object 
 14   taxonKey                            

In [52]:
e_data.info(verbose=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 61236 entries, 0 to 61235
Data columns (total 141 columns):
 #    Column                                 Dtype  
---   ------                                 -----  
 0    key                                    int64  
 1    datasetKey                             object 
 2    publishingOrgKey                       object 
 3    installationKey                        object 
 4    hostingOrganizationKey                 object 
 5    publishingCountry                      object 
 6    protocol                               object 
 7    lastCrawled                            object 
 8    lastParsed                             object 
 9    crawlId                                int64  
 10   extensions                             object 
 11   basisOfRecord                          object 
 12   occurrenceStatus                       object 
 13   classifications                        object 
 14   taxonKey                            

#### How do we know the occurrence is valid? 

- During kickoff the contact person from Nabu who participated suggested we only look at observations with pictures, but I haven't found a column that indicated that yet. 
- The column identificationVerificationStatus could be the one that indicates if the observation is valid or not, but it has lots of missing values. Also it would be good to learn if both categories valid and approved are indicators of valid data.

In [20]:
a_data.identificationVerificationStatus.value_counts()

identificationVerificationStatus
Approved | Expert | Evidence    1064
Valid                            244
Approved | Expert                128
Approved | Automated              24
Non réalisable                     4
Name: count, dtype: int64

In [4]:
e_data.identificationVerificationStatus.value_counts()

identificationVerificationStatus
Approved | Automated                                                                                                                        10337
Approved | Expert | Evidence                                                                                                                 7692
Approved | Expert                                                                                                                            1282
Valid                                                                                                                                           9
<a href='https://bee.questagame.com/#/expert/pastwork/367289/comments?search=true&sighting_source_id=367289&_k=1vxxpo'>Challenge ID?</a>        1
<a href='https://bee.questagame.com/#/expert/pastwork/441217/comments?search=true&sighting_source_id=441217&_k=1vxxpo'>Challenge ID?</a>        1
Name: count, dtype: int64

In [18]:
print('Asian h value counts for',a_data.scientificName.value_counts(),'\n\nEuropean h value counts for',e_data.scientificName.value_counts() )
 

Asian h value counts for scientificName
Vespa velutina Lepeletier, 1836             16689
Vespa velutina nigrithorax Buysson, 1905     1177
Name: count, dtype: int64 

European h value counts for scientificName
Vespa crabro Linnaeus                60614
Vespa crabro germana Christ, 1791      585
BOLD:ABA8441                            29
Vespa crabro crabro                      8
Name: count, dtype: int64


#### Missing values

In [ ]:
# if you want to check column by column for missing data, you can use the following code. I'm just checking the most relevant ones for now.

missing_data = a_data.isnull()
for column in missing_data.columns.values.tolist():
    print(column)
    print(missing_data[column].value_counts())
    print("")

In [29]:
# Asian
print('Country code: ',a_data.countryCode.isnull().value_counts(),
'\nDate Identified:',a_data.countryCode.isnull().value_counts(),
'\nIdentification Verification Status:',a_data.identificationVerificationStatus.isnull().value_counts(),
'\nDecimal Longitude:',a_data.decimalLongitude.isnull().value_counts(),
'\nDecimal Latitude:',a_data.decimalLatitude.isnull().value_counts())

Country code:  countryCode
False    17866
Name: count, dtype: int64 
Date Identified: countryCode
False    17866
Name: count, dtype: int64 
Identification Verification Status: identificationVerificationStatus
True     16402
False     1464
Name: count, dtype: int64 
Decimal Longitude: decimalLongitude
False    17863
True         3
Name: count, dtype: int64 
Decimal Latitude: decimalLatitude
False    17863
True         3
Name: count, dtype: int64


In [30]:
# European
print('Country code: ',e_data.countryCode.isnull().value_counts(),
'\nDate Identified:',e_data.countryCode.isnull().value_counts(),
'\nIdentification Verification Status:',e_data.identificationVerificationStatus.isnull().value_counts(),
'\nDecimal Longitude:',e_data.decimalLongitude.isnull().value_counts(),
'\nDecimal Latitude:',e_data.decimalLatitude.isnull().value_counts())

Country code:  countryCode
False    61236
Name: count, dtype: int64 
Date Identified: countryCode
False    61236
Name: count, dtype: int64 
Identification Verification Status: identificationVerificationStatus
True     41914
False    19322
Name: count, dtype: int64 
Decimal Longitude: decimalLongitude
False    61069
True       167
Name: count, dtype: int64 
Decimal Latitude: decimalLatitude
False    61069
True       167
Name: count, dtype: int64


### 3. Other observations

Country: 
- **countryCode** column corresponds to country when querying with GBIF
- **country** column is the full name of the country 
- **publishingCountry** we could exclude

Missing values:

- **IdentificationVerificationStatus** has several missing values for both species, if the field is the only way to verify if the observation is valid, we might have to drop the rows with missing values or imputate them
- **decimalLatitude** and **decimalLongitude** have a few missing values we should definitely drop

### 4. Summary of main questions and doubts

- Among all date columns, is **dateIdentified** the best one to use as main date indicator?
- **gadm** column seems to be geogrpahical at different levels of specificity, I'd like to confirm this with someone from Nabu. It could be an important column but maybe it's enough with **latitude** and **longitude**, the rest of the data could be derived from that. {'level0': {'gid': 'DEU', 'name': 'Germany'}, 'level1': {'gid': 'DEU.11_1', 'name': 'Rheinland-Pfalz'}, 'level2': {'gid': 'DEU.11.18_1', 'name': 'Ludwigshafen am Rhein'}, 'level3': {'gid': 'DEU.11.18.1_1', 'name': 'Ludwigshafen am Rhein'}}
- **scientificName** has different categories, of which I want to know if BOLD:ABA8441 for european hornet if invalid. If it is invalid, does it render the whole observation invalid?
- Since **scientificName** has different categories that I understand pertain to different subespecies, are they interested in us separating those subspecies for the analysis and questions?
- **identificationVerificationStatus** has several different values, are all valid? What do they mean?
- **coordinateUncertaintyInMeters** is this the error margin for latitude and longitude?
- What's the meaning of the variable **isSequenced**?

### 5. Next steps

- clarify questions about variables and data quality
- re query datasets using gbif library and see if they come back with discrepancies as well
- decide on which data to use and how to clean it using the information above and the result of clarifying questions with someone from nabu